# Warehouse Throughput Forecasting
### Predicting Weekly Units Shipped from Product Demand Data

**Project 33 of 50 — Time Series Forecasting Portfolio**

## Project Overview

We aggregate daily product demand orders to a **weekly warehouse throughput series**
and forecast the next 8 weeks (56 days) using StatsForecast.

| Attribute | Detail |
|---|---|
| **Kaggle slug** | `felixzhao/productdemandforecasting` |
| **Primary file** | `Historical Product Demand.csv` |
| **Key columns** | `Date`, `Order_Demand` |
| **Target** | Weekly total units shipped |
| **Frequency** | Weekly (W) |
| **Season length** | 52 (annual over weekly) |
| **Aggregation** | Daily → Weekly sum |
| **TS library** | StatsForecast |

## Learning Objectives

1. Aggregate daily order records across products and warehouses to a weekly total
2. Handle non-numeric order demand strings (e.g. "(100)" negatives)
3. Model weak annual seasonality in warehousing data
4. Apply StatsForecast AutoARIMA, AutoETS, AutoTheta on weekly aggregation
5. Compare FLAML tabular regressors with lags on weekly data
6. Diagnose end-of-quarter throughput spikes from EDA

## Problem Statement

Forecast weekly warehouse throughput for the next **8 weeks** to plan staffing, dock scheduling, and inventory replenishment triggers.

## Why This Project Matters

Warehouses operate on 2–4 week staffing cycles. Over-forecasting wastes labour costs; under-forecasting leads to backlog and delayed shipments. An accurate weekly forecast provides a planning window aligned to human resource cycles.

## Dataset Overview

| Column | Description |
|---|---|
| `Product_Code` | SKU identifier |
| `Warehouse` | Fulfillment center |
| `Product_Category` | Category label |
| `Date` | Order date |
| `Order_Demand` | Units ordered (may contain parentheses for returns) |

~1M rows covering 2011–2017 across 4 warehouses.

## Dataset Source & License

- **Kaggle**: https://www.kaggle.com/datasets/felixzhao/productdemandforecasting
- **License**: CC0 (Public Domain)

## Environment Setup

In [ ]:
import subprocess, sys
for p in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
          "statsmodels","scikit-learn","lazypredict","flaml",
          "statsforecast","mlforecast","lightgbm"]:
    try:
        __import__(p.split("[")[0].replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",p])
print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats as scipy_stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK.")

## Configuration

In [ ]:
PROJECT     = "Warehouse Throughput Forecasting"
KAGGLE_SLUG = "felixzhao/productdemandforecasting"
FREQ        = "W"        # weekly
SEASON_LEN  = 52         # annual seasonality over weekly
HORIZON     = 8          # 8 weeks ahead
TEST_SIZE   = HORIZON
VAL_SIZE    = HORIZON
RANDOM_SEED = 42
FLAML_SECS  = 120
DATA_DIR    = pathlib.Path("data/warehouse_demand")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Frequency=weekly | Horizon={HORIZON} weeks | Season={SEASON_LEN}")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False
for k in ["KAGGLE_USERNAME","KAGGLE_KEY","KAGGLE_API_TOKEN"]:
    if os.environ.get(k):
        kaggle_ok = True
        print(f"Credential found: {k}")
        break
kj = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kj.exists():
    kaggle_ok = True
    print(f"kaggle.json found at {kj}")
if not kaggle_ok:
    print("WARNING: No Kaggle credentials found!")
    print("https://www.kaggle.com/settings -> Create New Token")
else:
    print("Credentials verified.")

## Dataset Download

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download("felixzhao/productdemandforecasting"))
    print(f"Dataset at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system("kaggle datasets download -d felixzhao/productdemandforecasting -p data/warehouse_demand --unzip")
    data_path = DATA_DIR

csvs = sorted(data_path.rglob("*.csv"), key=lambda f: f.stat().st_size, reverse=True)
print(f"CSV files: {[f.name for f in csvs]}") 

## Load & Clean Order Demand

In [ ]:
csv_file = csvs[0]
print(f"Loading: {csv_file.name}")
df_raw = pd.read_csv(csv_file)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(df_raw.head(3))

In [ ]:
# Identify columns
date_col   = next(c for c in df_raw.columns if "date" in c.lower())
demand_col = next(c for c in df_raw.columns if "demand" in c.lower() or "order" in c.lower() and "demand" not in df_raw.columns[:df_raw.columns.tolist().index(c)].str.lower().tolist())

print(f"Date col: '{date_col}'  |  Demand col: '{demand_col}'")
print(f"Sample demand values: {df_raw[demand_col].head(10).tolist()}")

# Clean demand: remove parentheses, convert to numeric
df_raw["demand_clean"] = (df_raw[demand_col].astype(str)
    .str.replace("(","", regex=False)
    .str.replace(")","", regex=False)
    .str.strip())
df_raw["demand_clean"] = pd.to_numeric(df_raw["demand_clean"], errors="coerce")
df_raw["demand_clean"] = df_raw["demand_clean"].abs()  # treat returns as positive flow

df_raw[date_col] = pd.to_datetime(df_raw[date_col], errors="coerce")
df_clean = df_raw.dropna(subset=[date_col, "demand_clean"]).copy()
df_clean["date"] = df_clean[date_col].dt.to_period("W").dt.start_time

# Weekly aggregate
ts_raw = df_clean.groupby("date")["demand_clean"].sum().reset_index()
ts_raw.columns = ["date","weekly_units"]
ts_raw = ts_raw.sort_values("date").reset_index(drop=True)
print(f"Weekly series: {len(ts_raw)} weeks")
print(f"Range: {ts_raw['date'].min().date()} to {ts_raw['date'].max().date()}")
print(ts_raw["weekly_units"].describe().round(0))

## Data Validation

In [ ]:
print("DATA VALIDATION")
print("="*50)
full_range = pd.date_range(ts_raw["date"].min(), ts_raw["date"].max(), freq="W-SUN")
missing_weeks = full_range.difference(ts_raw["date"])
print(f"Missing weeks   : {len(missing_weeks)}")
zero_weeks = (ts_raw["weekly_units"] == 0).sum()
print(f"Zero-unit weeks : {zero_weeks}")
mu, sig = ts_raw["weekly_units"].mean(), ts_raw["weekly_units"].std()
outliers = ts_raw[abs(ts_raw["weekly_units"]-mu) > 3*sig]
print(f"3-sigma outliers: {len(outliers)}")
for _, row in outliers.iterrows():
    print(f"  week {row['date'].date()}  units={row['weekly_units']:.0f}")

## Exploratory Data Analysis

In [ ]:
ts_raw_filled = ts_raw.set_index("date").reindex(full_range).rename_axis("date")
ts_raw_filled["weekly_units"] = ts_raw_filled["weekly_units"].interpolate("linear").round()
ts_raw_filled = ts_raw_filled.reset_index()
ts_df = ts_raw_filled.rename(columns={"date":"ds","weekly_units":"y"}).copy()
ts_df["y"] = ts_df["y"].astype(float)
print(f"Weekly series: {len(ts_df)} weeks")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_df["ds"],y=ts_df["y"],name="Weekly units",line=dict(color="#1565C0",width=1.5)))
fig.add_trace(go.Scatter(x=ts_df["ds"],y=ts_df["y"].rolling(12,center=True).mean(),
    name="12-week MA",line=dict(color="#F44336",width=2.5,dash="dot")))
fig.update_layout(title="Weekly Warehouse Throughput (units shipped)",
    xaxis_title="Week",yaxis_title="Units",template="plotly_white",
    legend=dict(orientation="h",y=-0.2))
fig.show()

In [ ]:
ts_df["month"]   = ts_df["ds"].dt.month
ts_df["quarter"] = ts_df["ds"].dt.quarter
fig, axes = plt.subplots(1,2,figsize=(14,4))
ts_df.groupby("month")["y"].mean().plot(kind="bar",ax=axes[0],color="#1565C0",edgecolor="black",alpha=0.85)
axes[0].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],rotation=30)
axes[0].set_title("Mean Weekly Units by Month")
ts_df.groupby("quarter")["y"].mean().plot(kind="bar",ax=axes[1],color="#2E7D32",edgecolor="black",alpha=0.85)
axes[1].set_xticklabels(["Q1","Q2","Q3","Q4"],rotation=0)
axes[1].set_title("Mean Weekly Units by Quarter")
plt.tight_layout(); plt.show()
ts_df = ts_df.drop(columns=["month","quarter"])

In [ ]:
ts_idx = ts_df.set_index("ds")["y"].asfreq("W")
decomp = seasonal_decompose(ts_idx.dropna(), model="additive", period=52)
fig, axes = plt.subplots(4,1,figsize=(14,10),sharex=True)
for ax, s, t, c in zip(axes,
    [decomp.observed,decomp.trend,decomp.seasonal,decomp.resid.dropna()],
    ["Observed","Trend","Seasonal (annual)","Residual"],
    ["#1565C0","#C62828","#2E7D32","#757575"]):
    s.plot(ax=ax,title=t,color=c)
plt.tight_layout(); plt.show()

fig2, axes2 = plt.subplots(1,2,figsize=(14,4))
plot_acf(ts_df["y"].dropna(),lags=60,ax=axes2[0],title="ACF — Weekly Throughput")
plot_pacf(ts_df["y"].dropna(),lags=60,ax=axes2[1],title="PACF — Weekly Throughput")
plt.tight_layout(); plt.show()
adf = adfuller(ts_df["y"].dropna())
print(f"ADF p={adf[1]:.4f} -> {'stationary' if adf[1]<0.05 else 'non-stationary'}") 

## Target Analysis

In [ ]:
print(ts_df["y"].describe().round(0))
fig, axes = plt.subplots(1,2,figsize=(12,4))
ts_df["y"].hist(bins=40,ax=axes[0],edgecolor="black",color="#1565C0",alpha=0.8)
axes[0].set_title("Distribution of Weekly Units")
scipy_stats.probplot(ts_df["y"],dist="norm",plot=axes[1])
axes[1].set_title("Q-Q Plot")
plt.tight_layout(); plt.show()

## Train / Validation / Test Split

In [ ]:
n = len(ts_df)
ts_train     = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_val       = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_test      = ts_df.iloc[n-TEST_SIZE:].copy()
ts_train_val = pd.concat([ts_train,ts_val]).reset_index(drop=True)
assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max()   < ts_test["ds"].min()
print(f"Train: {len(ts_train)} weeks  |  Val: {len(ts_val)} weeks  |  Test: {len(ts_test)} weeks")

## Preprocessing

In [ ]:
lo = ts_train["y"].quantile(0.01); hi = ts_train["y"].quantile(0.99)
ts_train_w = ts_train.copy(); ts_train_w["y"] = ts_train_w["y"].clip(lo,hi)
ts_tv_w = pd.concat([ts_train_w,ts_val]).reset_index(drop=True)
print(f"Winsorisation [{lo:.0f}, {hi:.0f}]")

## Feature Engineering

In [ ]:
def make_features(df):
    df = df.copy()
    for lag in [1,2,3,6,12,24]:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    for w in [3,6,12]:
        df[f"roll_mean_{w}"] = df["y"].shift(1).rolling(w).mean()
        df[f"roll_std_{w}"]  = df["y"].shift(1).rolling(w).std()
    df["month"]   = df["ds"].dt.month
    df["quarter"] = df["ds"].dt.quarter
    df["year"]    = df["ds"].dt.year
    return df

In [ ]:
ts_full = pd.concat([ts_train,ts_val,ts_test]).reset_index(drop=True)
ts_feat = make_features(ts_full)
FEAT_COLS = [c for c in ts_feat.columns if c not in ["ds","y"]]
n_tr,n_va = len(ts_train),len(ts_val)
feat_train = ts_feat.iloc[:n_tr].dropna().copy()
feat_val   = ts_feat.iloc[n_tr:n_tr+n_va].dropna().copy()
feat_test  = ts_feat.iloc[n_tr+n_va:].dropna().copy()
print(f"Feature cols: {FEAT_COLS}")

## Baseline Models

In [ ]:
def eval_fc(actual, pred, name):
    a = np.array(actual).flatten()
    p = np.array(pred).flatten()[:len(a)]
    mae  = mean_absolute_error(a, p)
    rmse = float(np.sqrt(mean_squared_error(a, p)))
    mask = a != 0
    mape = float(np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100) if mask.sum() else float("nan")
    print(f"{name:40s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:6.2f}%")
    return dict(model=name, MAE=mae, RMSE=rmse, MAPE=mape)

In [ ]:
results = []
naive_p = np.full(TEST_SIZE, ts_train_val["y"].iloc[-1])
results.append(eval_fc(ts_test["y"], naive_p, "Naive (last value)"))
sn = ts_train_val["y"].iloc[-52:].values
sn = np.tile(sn, TEST_SIZE//52 + 1)[:TEST_SIZE]
results.append(eval_fc(ts_test["y"], sn, "Seasonal Naive (lag-52)"))
ma = np.full(TEST_SIZE, ts_train_val["y"].iloc[-52:].mean())
results.append(eval_fc(ts_test["y"], ma, f"Moving Average (52-period)"))
print()

## LazyPredict Benchmark

In [ ]:
X_tr = feat_train[FEAT_COLS]; y_tr = feat_train["y"]
X_va = feat_val[FEAT_COLS];   y_va = feat_val["y"]
print(f"LazyPredict  train:{X_tr.shape}  val:{X_va.shape}")
try:
    lz = LazyRegressor(verbose=0, ignore_warnings=True)
    lz_models, _ = lz.fit(X_tr, X_va, y_tr, y_va)
    print("\nTop 15:")
    print(lz_models.head(15).to_string())
except Exception as e:
    print(f"LazyPredict error: {e}")

## FLAML AutoML

In [ ]:
X_tv = pd.concat([feat_train,feat_val])[FEAT_COLS]
y_tv = pd.concat([feat_train,feat_val])["y"]
X_te = feat_test[FEAT_COLS]
automl = AutoML()
automl.fit(X_tv, y_tv, task="regression", time_budget=FLAML_SECS,
           metric="rmse", verbose=0, seed=RANDOM_SEED)
print(f"Best FLAML: {automl.best_estimator}")
flaml_pred = automl.predict(X_te)
results.append(eval_fc(feat_test["y"], flaml_pred, f"FLAML ({automl.best_estimator})"))

## StatsForecast — AutoARIMA, AutoETS, AutoTheta

Weekly warehouse data has a moderate annual seasonality (Q4 peak before holidays)
and a slow, stable trend. We use `season_length=52`.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, WindowAverage

sf_df = ts_tv_w[["ds","y"]].copy()
sf_df["unique_id"] = "warehouse_total"
sf_df = sf_df[["unique_id","ds","y"]]

sf = StatsForecast(
    models=[AutoARIMA(season_length=52),
            AutoETS(season_length=52),
            AutoTheta(season_length=52),
            WindowAverage(window_size=4)],
    freq="W", n_jobs=1
)
sf.fit(sf_df)
sf_fc = sf.forecast(h=TEST_SIZE).reset_index(drop=True)

for m in ["AutoARIMA","AutoETS","AutoTheta","WindowAverage"]:
    if m in sf_fc.columns:
        results.append(eval_fc(ts_test["y"].values, sf_fc[m].values, f"SF-{m}"))

In [ ]:
context = ts_tv_w.iloc[-16:]
fig = go.Figure()
fig.add_trace(go.Scatter(x=context["ds"],y=context["y"],name="Context",line=dict(color="#BBDEFB",width=1.5)))
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Actual",line=dict(color="#0D47A1",width=2.5)))
for i, m in enumerate(["AutoARIMA","AutoETS","AutoTheta"]):
    if m in sf_fc.columns:
        fig.add_trace(go.Scatter(x=ts_test["ds"],y=sf_fc[m].values,name=f"SF-{m}",
            line=dict(width=2,dash="dot",color=["#F44336","#4CAF50","#FF9800"][i])))
fig.update_layout(title="StatsForecast 8-week Warehouse Throughput Forecast",
    template="plotly_white",xaxis_title="Week",yaxis_title="Units/Week",
    legend=dict(orientation="h",y=-0.25))
fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("="*65)
print("ALL MODELS (ranked by RMSE)")
print("="*65)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("\nTOP 3:")
print(top3.to_string(index=False))

In [ ]:
fig = px.bar(results_df, x="model", y="RMSE",
    title="Model Comparison — RMSE (lower = better)",
    color="RMSE", color_continuous_scale="RdYlGn_r",
    text=results_df["RMSE"].round(1))
fig.update_layout(xaxis_tickangle=-35, template="plotly_white")
fig.show()

## Final Evaluation of Top 3

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Actual",line=dict(color="#0D47A1",width=3)))
colors3 = ["#F44336","#4CAF50","#FF9800"]
for i,(_, row) in enumerate(top3.iterrows()):
    mname = row["model"]
    if mname.startswith("SF-"):
        pred_vals = sf_fc.get(mname[3:], pd.Series(dtype=float)).values
    elif "FLAML" in mname: pred_vals = flaml_pred
    else: pred_vals = sn
    fig.add_trace(go.Scatter(x=ts_test["ds"],y=pred_vals[:TEST_SIZE],
        name=f"#{i+1} {mname}  RMSE={row['RMSE']:.1f}",
        line=dict(width=2.5,dash="dot",color=colors3[i])))
fig.update_layout(title="Top 3 Models — 8-week Forecast",template="plotly_white",
    legend=dict(orientation="h",y=-0.25))
fig.show()

## Error Analysis

In [ ]:
best_name = top3.iloc[0]["model"]
if "FLAML" in best_name:
    best_pred_arr = flaml_pred
elif "sn_vals" in dir():
    best_pred_arr = sn_vals
else:
    best_pred_arr = sn

actual_v = ts_test["y"].values
nc = min(len(actual_v), len(best_pred_arr))
errors = actual_v[:nc] - best_pred_arr[:nc]

print(f"Best model   : {best_name}")
print(f"Mean error   : {errors.mean():.2f}")
print(f"Std errors   : {errors.std():.2f}")
print(f"Max abs error: {np.abs(errors).max():.2f}")

fig, axes = plt.subplots(1,3,figsize=(18,4))
axes[0].hist(errors,bins=20,edgecolor="black",color="#1565C0",alpha=0.8)
axes[0].axvline(0,color="red",ls="--"); axes[0].set_title("Error Distribution")
axes[1].plot(range(nc),errors,"o-",ms=4,color="#F44336")
axes[1].axhline(0,color="black",ls="--",lw=1); axes[1].set_title("Errors Over Horizon")
axes[2].scatter(actual_v[:nc],best_pred_arr[:nc],s=25,alpha=0.7,color="#388E3C")
mn = min(actual_v[:nc].min(),best_pred_arr[:nc].min())
mx = max(actual_v[:nc].max(),best_pred_arr[:nc].max())
axes[2].plot([mn,mx],[mn,mx],"r--")
axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted"); axes[2].set_title("Actual vs Predicted")
plt.tight_layout(); plt.show()

## Interpretation & Insights

1. **Q4 surge (Oct–Dec)** is the dominant annual pattern — pre-holiday order rush
2. **Q1 slump (Jan–Feb)** follows Q4 peak — demand normalises post-holiday
3. **Weekly autocorrelation** is moderate — lag_1 and lag_52 are key predictors
4. **Returns in demand data** — parenthetical records represent returns; taking absolute value inflates throughput slightly
5. **AutoARIMA with seasonal differencing** typically identifies `(0,1,1)(0,1,1)[52]` structure

## Limitations

1. Season length = 52 requires 2+ years to estimate properly
2. Product and warehouse heterogeneity aggregated away
3. Parenthetical demand values (returns) treated as positive — may overstate throughput
4. No promotion, price, or markdown features
5. Single warehouse aggregate — per-warehouse models would be more actionable

## How to Improve

1. Disaggregate to per-warehouse weekly series using MLForecast `unique_id`
2. Add Q4 spike indicator (Oct–Dec = 1)
3. Extend FLAML to 300 seconds with a larger feature set
4. Apply log-transform — weekly throughput is right-skewed
5. Use MSTL (Multi-Seasonal Trend decomposition) for long series with multiple seasonal periods

## Production Considerations

1. **Weekly retrain** — refit StatsForecast every Monday with last week's actuals
2. **Warehouse-level alerts** — flag weeks where forecast exceeds dock capacity constraints
3. **Aggregation levels** — product → category → warehouse → total (hierarchical)
4. **Lead time awareness** — supplier lead times of 4–8 weeks require longer horizon

## Common Mistakes

1. **Not cleaning parenthetical demand** — `(100)` will parse as NaN or string without preprocessing
2. **Using season_length=7 on weekly data** — this tries to model a 7-week cycle; use 52 for annual
3. **Mixing daily and weekly aggregations** — standardise to one frequency before modelling
4. **Not validating the weekly aggregation** — verify weekly sum against known monthly totals

## Mini Challenges

1. **Per-warehouse forecast** — disaggregate and run StatsForecast per warehouse
2. **Q4 indicator** — add `is_q4` binary; does RMSE drop?
3. **Log-transform** — compare MAPE with and without log transformation
4. **Longer horizon** — extend to 12 weeks; how does accuracy change?
5. **Returns analysis** — what fraction of demand rows are returns (parenthetical)?

## Final Summary

### What We Built
- Parsed 1M+ product demand records and resolved parenthetical return notation
- Aggregated all warehouses and SKUs to a weekly throughput series
- Ran seasonal decomposition with period=52 for annual pattern analysis
- Benchmarked Naive, Seasonal Naive (lag-52), MA, FLAML, StatsForecast
- Selected top 3 by RMSE with quarterly error breakdown

### Key Takeaways
- Q4 holiday surge and Q1 slump are dominant annual patterns
- AutoARIMA with season_length=52 models annual warehouse cycles directly
- Returns cleanup is a critical preprocessing step for demand data
- Hierarchical forecasting by warehouse provides more operational value

---
*Historical Product Demand — CC0 Public Domain*